# Image and System Analysis | Division of Medical Radiation Physics | Stockholm University
```mehdi.astaraki@fysik.su.se```

# 1D Signals and Systems Fundamentals
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Astarakee/isa-su/blob/main/labs/01_SignalSystems.ipynb)

**Course**: Image and System Analysis

**Level**: Undergraduate / Graduate Computational Lab

**Target Audience**: Medical Physicists, Computational Researchers, Biomedical Engineers, Image and Signal Processing Students

**Author**: `Mehdi Astaraki`

---

## Overview & Learning Objectives
This Jupyter Notebook serves as an interactive computational lab manual for understanding the core concepts of **1D Signals and Linear Time-Invariant (LTI) Systems**.

By completing this notebook, you will learn to:
1. Distinguish mathematically and visually between **Continuous-Time (CT)** and **Discrete-Time (DT)** 1D signals.
2. Analyze **periodic vs. non-periodic** behavior in signals.
3. Synthesize and evaluate **fundamental benchmark signals** (Dirac Delta, Unit Step, Rectangular Pulse, Sinc).
4. Perform **even and odd signal decomposition** both analytically and numerically.
5. Characterize system properties: **Memory, Linearity, Time-Invariance, Causality, Stability (BIBO), and Invertibility** through explicit numerical experiments.
6. Perform time-domain transformations including **time-shifting**.
7. Understand 1D **convolution** conceptually, step-by-step computationally, and via standard libraries.
8. Verify key theoretical properties of linear convolution (Commutativity and Shift-Invariance).

---


In [ ]:
# Setup Environment and Libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

# Configure high-quality inline plotting
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.autolayout'] = True

print("Libraries imported successfully. NumPy version:", np.__version__)


---
## SECTION 1: Introduction to 1D Signals (Continuous vs. Discrete)

### Theoretical Background
A **1D Signal** is a scalar mathematical function representing how a dynamic physical quantity varies over a single independent variable, typically time $t$ or spatial position $x$. 
Real-world examples include:
- **Electrocardiogram (ECG)**: Electric potential of the heart measured over time.
- **Acoustic Audio**: Air pressure fluctuations captured by a microphone.
- **Financial Stock Indices**: Asset prices updated sequentially.

#### Mathematical Definitions:
1. **Continuous-Time (CT) Signal $x(t)$**: Defined for a continuous range of time $t \in \mathbb{R}$. The independent variable $t$ takes any real number.
2. **Discrete-Time (DT) Signal $x[n]$**: Defined only at discrete integer indices $n \in \mathbb{Z}$. It is obtained by sampling a continuous signal $x(t)$ at uniform sampling intervals $T_s$, such that:
   $$x[n] = x(t)\Big|_{t = n T_s} = x(n T_s)$$

#### Real-World Noise Intrusion:
Signals measured in real-world environments are inevitably corrupted by **Additive White Gaussian Noise (AWGN)**, modeled as:
$$x_{\text{noisy}}(t) = x(t) + \eta(t), \quad \text{where } \eta(t) \sim \mathcal{N}(0, \sigma^2)$$

Below, we simulate a pure continuous sinusoid $x(t) = A \sin(2\pi f t)$ and its noisy counterpart, alongside their discrete sampled representations.


In [ ]:
# Section 1: Code Implementation
# 1. Define time vectors
t_cont = np.linspace(0, 1.0, 1000)  # Dense grid simulating continuous time t
fs = 20                             # Sampling frequency (Hz)
Ts = 1.0 / fs                       # Sampling interval
n_disc = np.arange(0, int(1.0 * fs) + 1) # Discrete sample index n
t_disc = n_disc * Ts                # Discrete sampling times

# 2. Signal Parameters
freq = 3.0  # Frequency of 3 Hz
amp = 2.0   # Amplitude A = 2.0
noise_std = 0.5 # Standard deviation of Gaussian noise

# 3. Generate Signals
# Continuous-time signals
x_cont_pure = amp * np.sin(2 * np.pi * freq * t_cont)
np.random.seed(42) # For reproducible noise
noise_cont = np.random.normal(0, noise_std, size=t_cont.shape)
x_cont_noisy = x_cont_pure + noise_cont

# Discrete-time signals (sampled versions)
x_disc_pure = amp * np.sin(2 * np.pi * freq * t_disc)
noise_disc = np.random.normal(0, noise_std, size=t_disc.shape)
x_disc_noisy = x_disc_pure + noise_disc

# 4. Multi-panel Visualization
fig, axs = plt.subplots(2, 2, figsize=(13, 8))

# Subplot 1: Continuous Pure Sine
axs[0, 0].plot(t_cont, x_cont_pure, 'b-', linewidth=2, label=r'$x(t) = 2\sin(2\pi \cdot 3 t)$')
axs[0, 0].set_title(r"Continuous-Time Pure Signal $x(t)$", fontsize=12, fontweight='bold')
axs[0, 0].set_xlabel("Time $t$ (seconds)")
axs[0, 0].set_ylabel("Amplitude")
axs[0, 0].legend()

# Subplot 2: Discrete Sampled Pure Sine
axs[0, 1].stem(n_disc, x_disc_pure, linefmt='b-', markerfmt='bo', basefmt='k-', label=r'$x[n] = x(n T_s)$')
axs[0, 1].set_title(f"Discrete-Time Pure Signal $x[n]$ (Sampling rate $f_s = {fs}$ Hz)", fontsize=12, fontweight='bold')
axs[0, 1].set_xlabel("Sample Index $n$")
axs[0, 1].set_ylabel("Amplitude")
axs[0, 1].legend()

# Subplot 3: Continuous Noisy Sine
axs[1, 0].plot(t_cont, x_cont_noisy, 'r-', linewidth=1.5, alpha=0.85, label=r'$x(t) + \eta(t)$')
axs[1, 0].plot(t_cont, x_cont_pure, 'k--', linewidth=1, alpha=0.6, label='Clean Signal')
axs[1, 0].set_title("Continuous-Time Noisy Signal", fontsize=12, fontweight='bold')
axs[1, 0].set_xlabel("Time $t$ (seconds)")
axs[1, 0].set_ylabel("Amplitude")
axs[1, 0].legend()

# Subplot 4: Discrete Noisy Sine
axs[1, 1].stem(n_disc, x_disc_noisy, linefmt='r-', markerfmt='ro', basefmt='k-', label=r'$x[n] + \eta[n]$')
axs[1, 1].stem(n_disc, x_disc_pure, linefmt='k--', markerfmt='ko', basefmt='k-', label='Clean Samples')
axs[1, 1].set_title(r"Discrete-Time Noisy Signal $x[n]$", fontsize=12, fontweight='bold')
axs[1, 1].set_xlabel("Sample Index $n$")
axs[1, 1].set_ylabel("Amplitude")
axs[1, 1].legend()

plt.suptitle("Section 1: Continuous vs. Discrete 1D Signals", fontsize=14, fontweight='bold')
plt.show()


---
## SECTION 2: Periodic vs. Non-Periodic Signals

### Theoretical Background
Signals can be classified based on whether their temporal patterns repeat indefinitely at regular intervals.

#### 1. Periodic Signal:
A continuous signal $x(t)$ is **periodic** if there exists a positive real constant fundamental period $T > 0$ such that:
$$x(t) = x(t + T), \quad \forall t \in \mathbb{R}$$
The fundamental period $T_0$ is the smallest positive value of $T$ satisfying this equality.

#### 2. Non-Periodic (Aperiodic) Signal:
A signal $x(t)$ is **non-periodic** if no such constant $T > 0$ exists. A non-periodic signal typically represents a localized, finite-duration event (pulse) or a non-repeating transient process.

In the numerical implementation below, we contrast a periodic triangular wave of period $T = 2.0\text{ s}$ with a non-periodic single triangular pulse of duration $T_{\text{pulse}} = 2.0\text{ s}$.


In [ ]:
# Section 2: Code Implementation
t_sec2 = np.linspace(-4, 6, 1000)

# 1. Construct Periodic Triangular Wave using scipy.signal.sawtooth (width=0.5 yields symmetric triangle)
T_period = 2.0
freq_tri = 1.0 / T_period
# Sawtooth with width 0.5 ranges from -1 to 1; scaled to amplitude 1
x_periodic_tri = signal.sawtooth(2 * np.pi * freq_tri * t_sec2, width=0.5)

# 2. Construct Non-Periodic Single Triangular Pulse
# A single triangular pulse centered at t=0 with total base width 2.0 (from t=-1 to t=1)
T_pulse = 2.0
x_aperiodic_tri = np.maximum(0, 1 - np.abs(t_sec2 / (T_pulse / 2)))

# 3. Plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

# Periodic Plot
ax1.plot(t_sec2, x_periodic_tri, 'b-', linewidth=2, label='Periodic Triangular Wave')
ax1.axvline(0, color='gray', linestyle=':', alpha=0.6)
ax1.axvline(T_period, color='gray', linestyle=':', alpha=0.6)
ax1.annotate('', xy=(0, 1.1), xytext=(T_period, 1.1),
             arrowprops=dict(arrowstyle='<->', color='red', lw=1.5))
ax1.text(T_period/2, 1.15, f'Period $T = {T_period}$ s', color='red', fontsize=11, ha='center')
ax1.set_title(r"Periodic Signal ($x(t) = x(t + T)$)", fontsize=12, fontweight='bold')
ax1.set_xlabel("Time $t$ (seconds)")
ax1.set_ylabel("Amplitude")
ax1.set_ylim(-1.3, 1.4)
ax1.legend(loc='lower right')

# Non-Periodic Plot
ax2.plot(t_sec2, x_aperiodic_tri, 'g-', linewidth=2, label='Single Triangular Pulse')
ax2.axvline(-T_pulse/2, color='gray', linestyle=':', alpha=0.6)
ax2.axvline(T_pulse/2, color='gray', linestyle=':', alpha=0.6)
ax2.annotate('', xy=(-T_pulse/2, 1.1), xytext=(T_pulse/2, 1.1),
             arrowprops=dict(arrowstyle='<->', color='purple', lw=1.5))
ax2.text(0, 1.15, f'Pulse Duration $T_{{pulse}} = {T_pulse}$ s', color='purple', fontsize=11, ha='center')
ax2.set_title("Non-Periodic Signal (Isolated Transient Pulse)", fontsize=12, fontweight='bold')
ax2.set_xlabel("Time $t$ (seconds)")
ax2.set_ylabel("Amplitude")
ax2.set_ylim(-0.3, 1.4)
ax2.legend(loc='lower right')

plt.suptitle("Section 2: Periodic vs. Non-Periodic Signals", fontsize=14, fontweight='bold')
plt.show()


---
## SECTION 3: Fundamental Benchmark Signals

In signal processing, benchmark signals act as elementary building blocks to model dynamic inputs, characterize systems, and decompose complex signals.

### Mathematical Definitions:

1. **Dirac Delta Function $\delta(t)$ / Unit Impulse $\delta[n]$**:
   - Continuous approximation: A narrow pulse of width $\epsilon$ and height $1/\epsilon$, satisfying $\int_{-\infty}^{\infty} \delta(t) dt = 1$.
   - Discrete definition:
     $$\delta[n] = \begin{cases} 1, & n = 0 \\ 0, & n \neq 0 \end{cases}$$

2. **Unit Step Function $u(t)$ / $u[n]$**:
   - Models an instantaneous turn-on switch event at time 0:
     $$u(t) = \begin{cases} 1, & t \ge 0 \\ 0, & t < 0 \end{cases}, \quad u[n] = \begin{cases} 1, & n \ge 0 \\ 0, & n < 0 \end{cases}$$

3. **Rectangular Pulse Function $\text{rect}(t/T)$**:
   - Represents a windowing operation of total width $T$:
     $$\text{rect}\left(\frac{t}{T}\right) = \begin{cases} 1, & |t| \le T/2 \\ 0, & |t| > T/2 \end{cases}$$

4. **Normalized Sinc Function $\text{sinc}(t)$**:
   - Essential for ideal low-pass filtering and bandlimited interpolation:
     $$\text{sinc}(t) = \frac{\sin(\pi t)}{\pi t}, \quad \text{with } \text{sinc}(0) = 1$$

Below, we display a $4 \times 2$ grid comparing each benchmark signal across both continuous and discrete representations.


In [ ]:
# Section 3: Code Implementation
t_bench = np.linspace(-5, 5, 1000)
n_bench = np.arange(-5, 6)

# Continuous Benchmark Signals
# 1. Dirac Delta approximation (narrow pulse of width epsilon=0.1, area=1)
eps = 0.1
delta_cont = np.where(np.abs(t_bench) <= eps / 2, 1.0 / eps, 0.0)

# 2. Unit Step u(t)
u_cont = np.where(t_bench >= 0, 1.0, 0.0)

# 3. Rectangular Pulse rect(t/T) with T=2
T_rect = 2.0
rect_cont = np.where(np.abs(t_bench) <= T_rect / 2, 1.0, 0.0)

# 4. Sinc function (np.sinc computes sin(pi*x)/(pi*x))
sinc_cont = np.sinc(t_bench)

# Discrete Benchmark Signals
# 1. Discrete Impulse delta[n]
delta_disc = np.where(n_bench == 0, 1.0, 0.0)

# 2. Discrete Unit Step u[n]
u_disc = np.where(n_bench >= 0, 1.0, 0.0)

# 3. Discrete Rectangular Pulse rect[n] with width N_rect=2 (n in [-2, 2])
rect_disc = np.where(np.abs(n_bench) <= 2, 1.0, 0.0)

# 4. Discrete Sinc
sinc_disc = np.sinc(n_bench)

# Create 4x2 Grid Plot
fig, axs = plt.subplots(4, 2, figsize=(12, 12))

# Row 0: Impulse
axs[0, 0].plot(t_bench, delta_cont, 'b-', linewidth=2)
axs[0, 0].set_title(r"Continuous Dirac Delta $\delta(t)$ (Narrow Pulse Approx.)")
axs[0, 0].set_ylabel("Amplitude")

axs[0, 1].stem(n_bench, delta_disc, linefmt='b-', markerfmt='bo', basefmt='k-')
axs[0, 1].set_title(r"Discrete Impulse $\delta[n]$")
axs[0, 1].set_ylabel("Amplitude")

# Row 1: Unit Step
axs[1, 0].plot(t_bench, u_cont, 'g-', linewidth=2)
axs[1, 0].set_title(r"Continuous Unit Step $u(t)$")
axs[1, 0].set_ylabel("Amplitude")

axs[1, 1].stem(n_bench, u_disc, linefmt='g-', markerfmt='go', basefmt='k-')
axs[1, 1].set_title(r"Discrete Unit Step $u[n]$")
axs[1, 1].set_ylabel("Amplitude")

# Row 2: Rectangular Pulse
axs[2, 0].plot(t_bench, rect_cont, 'm-', linewidth=2)
axs[2, 0].set_title(r"Continuous Rectangular Pulse $\text{rect}(t/2)$")
axs[2, 0].set_ylabel("Amplitude")

axs[2, 1].stem(n_bench, rect_disc, linefmt='m-', markerfmt='mo', basefmt='k-')
axs[2, 1].set_title(r"Discrete Rectangular Pulse $\text{rect}[n/2]$")
axs[2, 1].set_ylabel("Amplitude")

# Row 3: Sinc Function
axs[3, 0].plot(t_bench, sinc_cont, 'c-', linewidth=2)
axs[3, 0].set_title(r"Continuous Sinc Function $\text{sinc}(t)$")
axs[3, 0].set_xlabel("Time $t$")
axs[3, 0].set_ylabel("Amplitude")

axs[3, 1].stem(n_bench, sinc_disc, linefmt='c-', markerfmt='co', basefmt='k-')
axs[3, 1].set_title(r"Discrete Sinc Function $\text{sinc}[n]$")
axs[3, 1].set_xlabel("Sample Index $n$")
axs[3, 1].set_ylabel("Amplitude")

plt.suptitle("Section 3: Fundamental Benchmark Signals (Continuous vs. Discrete)", fontsize=14, fontweight='bold')
plt.show()


---
## SECTION 4: Signal Decomposition — Even and Odd Components

### Theoretical Background
Any arbitrary continuous-time signal $x(t)$ defined over a symmetric time interval around $t=0$ can be uniquely decomposed into the sum of an **even component** $x_e(t)$ and an **odd component** $x_o(t)$:

$$x(t) = x_e(t) + x_o(t)$$

#### Formulations:
- **Even Signal Property**: $x_e(t) = x_e(-t)$ (Symmetric across vertical axis).
  $$x_e(t) = \frac{x(t) + x(-t)}{2}$$
- **Odd Signal Property**: $x_o(t) = -x_o(-t)$ (Anti-symmetric through origin).
  $$x_o(t) = \frac{x(t) - x(-t)}{2}$$

#### Numerical Example:
Consider an asymmetric damped exponential pulse starting at $t=0$:
$$x(t) = e^{-0.5 t} \cos(2\pi t) \cdot u(t)$$

We compute $x(-t)$, extract $x_e(t)$ and $x_o(t)$, and numerically verify that $x_e(t) + x_o(t) \equiv x(t)$.


In [ ]:
# Section 4: Code Implementation
t_sec4 = np.linspace(-4, 4, 1000)

# Define asymmetric continuous signal x(t) = exp(-0.5*t) * cos(2*pi*t) * u(t)
u_sec4 = np.where(t_sec4 >= 0, 1.0, 0.0)
x_orig = np.exp(-0.5 * t_sec4) * np.cos(2 * np.pi * t_sec4) * u_sec4

# Time reversal: x(-t) is obtained by sampling at -t_sec4
u_neg = np.where(-t_sec4 >= 0, 1.0, 0.0)
x_reversed = np.exp(-0.5 * (-t_sec4)) * np.cos(2 * np.pi * (-t_sec4)) * u_neg

# Compute Even and Odd Components
x_even = 0.5 * (x_orig + x_reversed)
x_odd  = 0.5 * (x_orig - x_reversed)

# Reconstructed signal
x_reconstructed = x_even + x_odd

# Numerical error verification
max_err = np.max(np.abs(x_orig - x_reconstructed))
print(f"Max Absolute Error between x(t) and [x_e(t) + x_o(t)]: {max_err:.2e}")

# Plotting
fig, axs = plt.subplots(2, 2, figsize=(13, 8))

# Subplot 1: Original Signal
axs[0, 0].plot(t_sec4, x_orig, 'b-', linewidth=2, label=r'$x(t) = e^{-0.5t}\cos(2\pi t)u(t)$')
axs[0, 0].set_title(r"Original Asymmetric Signal $x(t)$", fontsize=12, fontweight='bold')
axs[0, 0].set_xlabel("Time $t$")
axs[0, 0].set_ylabel("Amplitude")
axs[0, 0].legend()

# Subplot 2: Time-Reversed Signal
axs[0, 1].plot(t_sec4, x_reversed, 'orange', linewidth=2, label=r'$x(-t)$')
axs[0, 1].set_title(r"Time-Reversed Signal $x(-t)$", fontsize=12, fontweight='bold')
axs[0, 1].set_xlabel("Time $t$")
axs[0, 1].set_ylabel("Amplitude")
axs[0, 1].legend()

# Subplot 3: Even and Odd Components
axs[1, 0].plot(t_sec4, x_even, 'g-', linewidth=2, label=r'Even Component $x_e(t)$')
axs[1, 0].plot(t_sec4, x_odd, 'r--', linewidth=2, label=r'Odd Component $x_o(t)$')
axs[1, 0].set_title(r"Decomposed Components $x_e(t)$ and $x_o(t)$", fontsize=12, fontweight='bold')
axs[1, 0].set_xlabel("Time $t$")
axs[1, 0].set_ylabel("Amplitude")
axs[1, 0].legend()

# Subplot 4: Reconstructed vs Original
axs[1, 1].plot(t_sec4, x_orig, 'b-', linewidth=3, alpha=0.6, label='Original $x(t)$')
axs[1, 1].plot(t_sec4, x_reconstructed, 'k:', linewidth=2, label=r'Reconstructed $x_e(t) + x_o(t)$')
axs[1, 1].set_title(r"Verification: $x(t) \equiv x_e(t) + x_o(t)$", fontsize=12, fontweight='bold')
axs[1, 1].set_xlabel("Time $t$")
axs[1, 1].set_ylabel("Amplitude")
axs[1, 1].legend()

plt.suptitle("Section 4: Even and Odd Signal Decomposition", fontsize=14, fontweight='bold')
plt.show()


---
## SECTION 5: Fundamental System Properties

A **System** $T\{\cdot\}$ is a mathematical operator mapping an input signal $x(t)$ or $x[n]$ to an output signal $y(t) = T\{x(t)\}$. 

Below, we explore 6 fundamental system properties, each illustrated with **two distinct numerical examples** showing input-output behavior.

---

### 1. Memory vs. Memoryless Systems
- **Memoryless System**: Output at time $t_0$ depends strictly on the input at the exact same instant $t_0$.  
  - *Example 1 (Memoryless)*: Scaling amplifier $y(t) = 3 x(t)$.
- **System with Memory**: Output at time $t_0$ depends on past or future inputs.  
  - *Example 2 (With Memory)*: Running discrete accumulator $y[n] = \sum_{k=-\infty}^n x[k]$.


In [ ]:
# Section 5.1: Memory vs. Memoryless Systems
t_s5 = np.linspace(0, 5, 200)
n_s5 = np.arange(0, 20)

# Input signal: Pulse input
x_pulse_ct = np.where((t_s5 >= 1) & (t_s5 <= 3), 2.0, 0.0)
x_pulse_dt = np.where((n_s5 >= 3) & (n_s5 <= 8), 2.0, 0.0)

# Example 1: Memoryless System y(t) = 3 * x(t)
y_memoryless = 3.0 * x_pulse_ct

# Example 2: System with Memory (Accumulator) y[n] = sum_{k=0}^n x[k]
y_memory = np.cumsum(x_pulse_dt)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(t_s5, x_pulse_ct, 'b--', label='Input $x(t)$')
ax1.plot(t_s5, y_memoryless, 'r-', linewidth=2, label='Output $y(t) = 3 x(t)$')
ax1.set_title("Memoryless System: Scaling Amplifier")
ax1.set_xlabel("Time $t$")
ax1.set_ylabel("Amplitude")
ax1.legend()

ax2.stem(n_s5, x_pulse_dt, linefmt='b--', markerfmt='bo', label='Input $x[n]$')
ax2.stem(n_s5, y_memory, linefmt='r-', markerfmt='rs', label=r'Output $y[n] = \sum x[k]$')
ax2.set_title("System with Memory: Discrete Accumulator")
ax2.set_xlabel("Sample $n$")
ax2.set_ylabel("Amplitude")
ax2.legend()

plt.suptitle("5.1: Memory vs. Memoryless Systems", fontsize=13, fontweight='bold')
plt.show()


---
### 2. Linearity (Superposition Principle)
A system is **Linear** if it satisfies both **Additivity** and **Homogeneity**:
$$T\{a x_1(t) + b x_2(t)\} = a T\{x_1(t)\} + b T\{x_2(t)\}$$

- *Example 1 (Linear System)*: $y(t) = 2 x(t) + 3 x(t-1)$
- *Example 2 (Non-linear System)*: $y(t) = [x(t)]^2$


In [ ]:
# Section 5.2: Linearity
t_lin = np.linspace(0, 4, 400)
dt = t_lin[1] - t_lin[0]
shift_samples = int(1.0 / dt)

# Define two inputs
x1 = np.sin(2 * np.pi * 1.0 * t_lin)
x2 = np.where((t_lin >= 1) & (t_lin <= 3), 1.0, 0.0)
a, b = 2.0, 0.5

# Combined input
x_combo = a * x1 + b * x2

# System 1: Linear System y(t) = 2*x(t) + 3*x(t-1)
def sys_linear(x):
    x_shift = np.roll(x, shift_samples)
    x_shift[:shift_samples] = 0
    return 2.0 * x + 3.0 * x_shift

# System 2: Non-Linear System y(t) = [x(t)]^2
def sys_nonlinear(x):
    return x**2

# Test Linear System
y_lin_combo = sys_linear(x_combo)
y_lin_sep   = a * sys_linear(x1) + b * sys_linear(x2)
err_lin = np.max(np.abs(y_lin_combo - y_lin_sep))

# Test Non-Linear System
y_nonlin_combo = sys_nonlinear(x_combo)
y_nonlin_sep   = a * sys_nonlinear(x1) + b * sys_nonlinear(x2)
err_nonlin = np.max(np.abs(y_nonlin_combo - y_nonlin_sep))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(t_lin, y_lin_combo, 'b-', linewidth=2.5, label=r'$T\{a x_1 + b x_2\}$')
ax1.plot(t_lin, y_lin_sep, 'r--', linewidth=1.5, label=r'$a T\{x_1\} + b T\{x_2\}$')
ax1.set_title(f"Linear System (Max Diff = {err_lin:.1e})")
ax1.set_xlabel("Time $t$")
ax1.set_ylabel("Amplitude")
ax1.legend()

ax2.plot(t_lin, y_nonlin_combo, 'b-', linewidth=2.5, label=r'$T\{a x_1 + b x_2\}$')
ax2.plot(t_lin, y_nonlin_sep, 'r--', linewidth=1.5, label=r'$a T\{x_1\} + b T\{x_2\}$')
ax2.set_title(f"Non-Linear System $[x(t)]^2$ (Max Diff = {err_nonlin:.2f})")
ax2.set_xlabel("Time $t$")
ax2.set_ylabel("Amplitude")
ax2.legend()

plt.suptitle("5.2: Linearity Verification", fontsize=13, fontweight='bold')
plt.show()


---
### 3. Time-Invariance (Shift-Invariance)
A system is **Time-Invariant** if a time shift in the input signal produces an identical time shift in the output signal:
$$x(t - t_0) \implies y(t - t_0)$$

- *Example 1 (Time-Invariant)*: $y(t) = \sin(x(t))$
- *Example 2 (Time-Varying)*: $y(t) = t \cdot x(t)$


In [ ]:
# Section 5.3: Time-Invariance
t_ti = np.linspace(0, 5, 500)
dt = t_ti[1] - t_ti[0]
t0 = 1.0 # 1 second delay
shift_n = int(t0 / dt)

# Original input x(t) and shifted input x(t - t0)
x_orig = np.where((t_ti >= 0.5) & (t_ti <= 1.5), 1.0, 0.0)
x_shifted = np.roll(x_orig, shift_n)
x_shifted[:shift_n] = 0

# System 1: Time-Invariant y(t) = sin(x(t))
y1_orig = np.sin(x_orig)
y1_orig_shifted = np.roll(y1_orig, shift_n)
y1_orig_shifted[:shift_n] = 0
y1_from_shifted_input = np.sin(x_shifted)
err_ti1 = np.max(np.abs(y1_orig_shifted - y1_from_shifted_input))

# System 2: Time-Varying y(t) = t * x(t)
y2_orig = t_ti * x_orig
y2_orig_shifted = np.roll(y2_orig, shift_n)
y2_orig_shifted[:shift_n] = 0
y2_from_shifted_input = t_ti * x_shifted
err_ti2 = np.max(np.abs(y2_orig_shifted - y2_from_shifted_input))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(t_ti, y1_orig_shifted, 'b-', linewidth=2.5, label='Shifted Output $y(t-t_0)$')
ax1.plot(t_ti, y1_from_shifted_input, 'r--', linewidth=1.5, label=r'Output from Shifted Input $T\{x(t-t_0)\}$')
ax1.set_title(f"Time-Invariant System $\sin(x(t))$ (Diff = {err_ti1:.1e})")
ax1.set_xlabel("Time $t$")
ax1.legend()

ax2.plot(t_ti, y2_orig_shifted, 'b-', linewidth=2.5, label='Shifted Output $y(t-t_0)$')
ax2.plot(t_ti, y2_from_shifted_input, 'r--', linewidth=1.5, label=r'Output from Shifted Input $T\{x(t-t_0)\}$')
ax2.set_title(f"Time-Varying System $t \cdot x(t)$ (Diff = {err_ti2:.2f})")
ax2.set_xlabel("Time $t$")
ax2.legend()

plt.suptitle("5.3: Time-Invariance Verification", fontsize=13, fontweight='bold')
plt.show()


---
### 4. Causality
A system is **Causal** if the output $y[n_0]$ at any sample $n_0$ depends strictly on present and past inputs $x[n]$ for $n \le n_0$, but never on future inputs $n > n_0$.

- *Example 1 (Causal System)*: Moving average / FIR filter $y[n] = 0.5 x[n] + 0.5 x[n-1]$.
- *Example 2 (Non-Causal System)*: Ideal look-ahead system $y[n] = 0.5 x[n+1] + 0.5 x[n]$.


In [ ]:
# Section 5.4: Causality
n_causal = np.arange(0, 10)
x_step = np.where(n_causal >= 4, 1.0, 0.0) # Step turns on at n=4

# Example 1: Causal y[n] = 0.5 * x[n] + 0.5 * x[n-1]
x_prev = np.roll(x_step, 1)
x_prev[0] = 0
y_causal = 0.5 * x_step + 0.5 * x_prev

# Example 2: Non-Causal y[n] = 0.5 * x[n+1] + 0.5 * x[n]
x_next = np.roll(x_step, -1)
x_next[-1] = 0
y_noncausal = 0.5 * x_next + 0.5 * x_step

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.stem(n_causal, x_step, linefmt='k--', markerfmt='ko', label='Input $x[n]$ (Starts at n=4)')
ax1.stem(n_causal, y_causal, linefmt='g-', markerfmt='go', label='Output $y[n]$ (Starts at n=4)')
ax1.set_title("Causal System: $y[n] = 0.5 x[n] + 0.5 x[n-1]$")
ax1.set_xlabel("Sample $n$")
ax1.set_ylabel("Amplitude")
ax1.legend()

ax2.stem(n_causal, x_step, linefmt='k--', markerfmt='ko', label='Input $x[n]$ (Starts at n=4)')
ax2.stem(n_causal, y_noncausal, linefmt='r-', markerfmt='rs', label='Output $y[n]$ (Starts at n=3!)')
ax2.set_title("Non-Causal System: $y[n] = 0.5 x[n+1] + 0.5 x[n]$")
ax2.set_xlabel("Sample $n$")
ax2.set_ylabel("Amplitude")
ax2.legend()

plt.suptitle("5.4: Causality Comparison", fontsize=13, fontweight='bold')
plt.show()


---
### 5. Bounded-Input Bounded-Output (BIBO) Stability
A system is **BIBO Stable** if every bounded input $|x(t)| \le M_x < \infty$ produces a bounded output $|y(t)| \le M_y < \infty$.

- *Example 1 (Stable System)*: Nonlinear squashing $y(t) = \tanh(x(t))$ or $y(t) = e^{-|x(t)|}$.
- *Example 2 (Unstable System)*: Unbounded accumulator with non-zero DC step input $y[n] = y[n-1] + x[n]$.


In [ ]:
# Section 5.5: BIBO Stability
n_stab = np.arange(0, 30)
x_bounded = np.ones_like(n_stab, dtype=float) # Bounded constant input x[n] = 1

# Example 1: Stable System y[n] = tanh(x[n])
y_stable = np.tanh(x_bounded)

# Example 2: Unstable Accumulation System y[n] = sum_{k=0}^n x[k]
y_unstable = np.cumsum(x_bounded)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(n_stab, x_bounded, 'k--', label='Bounded Input $x[n] = 1$')
ax1.plot(n_stab, y_stable, 'g-o', label=r'Output $y[n] = \tanh(x[n]) \leq 1$')
ax1.set_title("BIBO Stable System (Output Remains Bounded)")
ax1.set_xlabel("Sample $n$")
ax1.set_ylabel("Amplitude")
ax1.set_ylim(0, 2)
ax1.legend()

ax2.plot(n_stab, x_bounded, 'k--', label='Bounded Input $x[n] = 1$')
ax2.plot(n_stab, y_unstable, 'r-s', label=r'Output $y[n] = \sum x[k] \to \infty$')
ax2.set_title("BIBO Unstable System (Unbounded Growth)")
ax2.set_xlabel("Sample $n$")
ax2.set_ylabel("Amplitude")
ax2.legend()

plt.suptitle("5.5: BIBO Stability", fontsize=13, fontweight='bold')
plt.show()


---
### 6. Invertibility and System Inverse
A system is **Invertible** if distinct inputs always yield distinct outputs, allowing an inverse system $T^{-1}$ to uniquely recover $x(t)$ from $y(t)$:
$$T^{-1}\{T\{x(t)\}\}= x(t)$$

- *Example 1 (Invertible System)*: $y(t) = 2 x(t) \implies x(t) = 0.5 y(t)$.
- *Example 2 (Non-Invertible System)*: $y(t) = [x(t)]^2$ (Sign information is lost).


In [ ]:
# Section 5.6: Invertibility
t_inv = np.linspace(-2, 2, 200)
x_inv_test = np.sin(2 * np.pi * t_inv) # Contains both positive and negative values

# Example 1: Invertible y(t) = 2 * x(t)
y_invertible = 2.0 * x_inv_test
x_recovered = 0.5 * y_invertible

# Example 2: Non-Invertible y(t) = x^2(t)
y_noninvertible = x_inv_test**2
# Attempting recovery via sqrt loses sign of negative values:
x_ambiguous = np.sqrt(y_noninvertible)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(t_inv, x_inv_test, 'b-', linewidth=2, label='Original Input $x(t)$')
ax1.plot(t_inv, x_recovered, 'r--', linewidth=2, label='Recovered $x(t) = 0.5 y(t)$')
ax1.set_title("Invertible System: $y(t) = 2 x(t)$")
ax1.set_xlabel("Time $t$")
ax1.set_ylabel("Amplitude")
ax1.legend()

ax2.plot(t_inv, x_inv_test, 'b-', linewidth=2, label='Original Input $x(t)$')
ax2.plot(t_inv, x_ambiguous, 'r--', linewidth=2, label=r'Attempted Recovery $\sqrt{y(t)}$')
ax2.set_title("Non-Invertible System: $y(t) = x^2(t)$ (Sign Loss)")
ax2.set_xlabel("Time $t$")
ax2.set_ylabel("Amplitude")
ax2.legend()

plt.suptitle("5.6: Invertibility and Inverse Systems", fontsize=13, fontweight='bold')
plt.show()


---
## SECTION 6: Signal Transformations — Time Shifting

### Theoretical Background
Time shifting modifies the temporal alignment of a signal without altering its waveform amplitude or shape:
$$y(t) = x(t - t_0)$$

- **Time Delay ($t_0 > 0$)**: The signal $x(t - t_0)$ is shifted to the **right** along the time axis (occurs later in time).
- **Time Advance ($t_0 < 0$)**: The signal $x(t + |t_0|)$ is shifted to the **left** along the time axis (occurs earlier in time).

Below, we take an asymmetric trapezoidal pulse $x(t)$ and plot $x(t)$, delayed signal $x(t - 3)$, and advanced signal $x(t + 3)$ on the same axes.


In [ ]:
# Section 6: Code Implementation
t_sec6 = np.linspace(-6, 8, 1000)

def asymmetric_trapezoid(t):
    # Generates an asymmetric trapezoidal pulse centered between t=0 and t=2
    y = np.zeros_like(t)
    # Ramp up from t=0 to t=0.5
    ramp_up = (t >= 0) & (t < 0.5)
    y[ramp_up] = t[ramp_up] / 0.5
    # Flat top from t=0.5 to t=1.5
    flat = (t >= 0.5) & (t < 1.5)
    y[flat] = 1.0
    # Ramp down from t=1.5 to t=2.5
    ramp_down = (t >= 1.5) & (t < 2.5)
    y[ramp_down] = 1.0 - (t[ramp_down] - 1.5) / 1.0
    return y

# Compute original, delayed, and advanced pulses
x_base = asymmetric_trapezoid(t_sec6)
x_delay = asymmetric_trapezoid(t_sec6 - 3.0)   # Shifted right by 3
x_advance = asymmetric_trapezoid(t_sec6 + 3.0) # Shifted left by 3

# Plotting
plt.figure(figsize=(12, 5))
plt.plot(t_sec6, x_base, 'b-', linewidth=2.5, label='Original Signal $x(t)$')
plt.plot(t_sec6, x_delay, 'r--', linewidth=2.0, label='Delayed Signal $x(t - 3)$ (Right Shift)')
plt.plot(t_sec6, x_advance, 'g-.', linewidth=2.0, label='Advanced Signal $x(t + 3)$ (Left Shift)')

plt.axvline(0, color='gray', linestyle=':', alpha=0.5)
plt.title("Section 6: Signal Transformations — Time Delay vs. Time Advance", fontsize=13, fontweight='bold')
plt.xlabel("Time $t$ (seconds)")
plt.ylabel("Amplitude")
plt.ylim(-0.2, 1.3)
plt.legend(fontsize=11)
plt.show()


---
## SECTION 7: 1D Signal Convolution

Convolution is the single most fundamental mathematical operation in LTI system theory. The output $y(t)$ of any continuous LTI system with impulse response $h(t)$ to an input $x(t)$ is given by:

$$y(t) = (x * h)(t) = \int_{-\infty}^{\infty} x(\tau) \, h(t - \tau) \, d\tau$$

For discrete-time systems:
$$y[n] = (x * h)[n] = \sum_{k=-\infty}^{\infty} x[k] \, h[n - k]$$

### Conceptual Interpretation (The 4 Steps of Convolution):
1. **Fold (Time-Reversal)**: Reflect the impulse response $h(\tau) \to h(-\tau)$.
2. **Shift**: Shift the reversed function by lag $t \to h(t - \tau)$.
3. **Multiply**: Compute the pointwise product $x(\tau) \cdot h(t - \tau)$.
4. **Integrate / Accumulate**: Compute the total area under the product curve to get $y(t)$.

---

### Part 1: Step-by-Step Stepwise Manual Implementation & Visualization


In [ ]:
# Section 7 Part 1: Step-by-Step Convolution Visualization
# Define time grid tau for integration
tau = np.linspace(-3, 6, 1000)
d_tau = tau[1] - tau[0]

# Define input signal x(tau) = rect(tau - 0.5) [width 1]
x_tau = np.where((tau >= 0) & (tau <= 1), 1.0, 0.0)

# Define impulse response h(tau) = triangular pulse [width 1]
h_tau = np.maximum(0, 1 - np.abs(tau - 0.5) * 2)

# Selected lag t for step visualization
t_lag = 1.2

# Shifted and reversed impulse response h(t_lag - tau)
h_shifted_flipped = np.maximum(0, 1 - np.abs((t_lag - tau) - 0.5) * 2)

# Pointwise product
product_tau = x_tau * h_shifted_flipped

# Full convolution accumulation across all lags t
t_conv_grid = np.linspace(-1, 5, 300)
y_full = np.zeros_like(t_conv_grid)
for idx, t_val in enumerate(t_conv_grid):
    h_shift_temp = np.maximum(0, 1 - np.abs((t_val - tau) - 0.5) * 2)
    y_full[idx] = np.sum(x_tau * h_shift_temp) * d_tau

# 4-Panel Visualization
fig, axs = plt.subplots(2, 2, figsize=(13, 8))

# Panel 1: Input Signals x(tau) and h(tau)
axs[0, 0].plot(tau, x_tau, 'b-', linewidth=2, label=r'Input $x(\tau)$ (Rect)')
axs[0, 0].plot(tau, h_tau, 'g--', linewidth=2, label=r'Impulse Response $h(\tau)$ (Triangle)')
axs[0, 0].set_title(r"Step 1: Input Signals in Dummy Variable $\tau$ Domain")
axs[0, 0].set_xlabel(r"$\tau$")
axs[0, 0].legend()

# Panel 2: Folded & Shifted h(t - tau)
axs[0, 1].plot(tau, x_tau, 'b-', linewidth=1.5, alpha=0.5, label=r'$x(\tau)$')
axs[0, 1].plot(tau, h_shifted_flipped, 'r-', linewidth=2, label=f'Shifted & Flipped $h({t_lag} - \\tau)$')
axs[0, 1].set_title(f"Step 2: Time-Reverse and Shift to Lag $t = {t_lag}$")
axs[0, 1].set_xlabel(r"$\tau$")
axs[0, 1].legend()

# Panel 3: Pointwise Product & Overlap Integration Area
axs[1, 0].plot(tau, product_tau, 'm-', linewidth=2, label=r'$x(\tau) \cdot h(t-\tau)$')
axs[1, 0].fill_between(tau, product_tau, color='magenta', alpha=0.3, label='Overlap Area (Integral)')
axs[1, 0].set_title(f"Step 3: Pointwise Product Area at $t = {t_lag}$")
axs[1, 0].set_xlabel(r"$\tau$")
axs[1, 0].legend()

# Panel 4: Convolution Result Output y(t)
axs[1, 1].plot(t_conv_grid, y_full, 'k-', linewidth=2.5, label=r'$y(t) = (x * h)(t)$')
axs[1, 1].axvline(t_lag, color='magenta', linestyle='--', label=f'Current Lag $t={t_lag}$')
axs[1, 1].plot(t_lag, np.sum(product_tau)*d_tau, 'mo', markersize=8)
axs[1, 1].set_title(r"Step 4: Accumulated Convolution Result $y(t)$")
axs[1, 1].set_xlabel("Time $t$")
axs[1, 1].legend()

plt.suptitle("Section 7.1: Step-by-Step Manual 1D Convolution Mechanics", fontsize=14, fontweight='bold')
plt.show()


---
### Part 2: Built-in Convolution Examples

We now use standard library functions (`np.convolve`) to evaluate 5 essential benchmark convolution pairs.
1. `conv(Triangle, Rectangular)`
2. `conv(Rectangular, Rectangular)` with equal widths
3. `conv(Rectangular, Rectangular)` with unequal widths
4. `conv(Truncated Sine, Delta(t))` — impulse response preservation
5. `conv(Truncated Sine, Delta(t - 10))` — shifted impulse response delay


In [ ]:
# Section 7 Part 2: Built-in Convolution Benchmarks
dt = 0.01

# Benchmark 1: conv(Triangle, Rectangular)
t_sig = np.linspace(-2, 3, 500)
x_tri = np.maximum(0, 1 - np.abs(t_sig)) # Triangle width 2
h_rect = np.where((t_sig >= 0) & (t_sig <= 1), 1.0, 0.0) # Rect width 1
y_tri_rect = np.convolve(x_tri, h_rect, mode='full') * dt
t_y1 = np.linspace(2*t_sig[0], 2*t_sig[-1], len(y_tri_rect))

# Benchmark 2: conv(Rectangular, Rectangular) Equal Widths (width = 2)
x_rect1 = np.where(np.abs(t_sig) <= 1.0, 1.0, 0.0)
y_rect_equal = np.convolve(x_rect1, x_rect1, mode='full') * dt

# Benchmark 3: conv(Rectangular, Rectangular) Unequal Widths (width1 = 2, width2 = 1)
h_rect_short = np.where(np.abs(t_sig) <= 0.5, 1.0, 0.0)
y_rect_unequal = np.convolve(x_rect1, h_rect_short, mode='full') * dt

# Benchmark 4 & 5: conv(Truncated Sine, Delta(t)) and conv(Truncated Sine, Delta(t - t0))
n_dt = np.arange(-5, 25)
x_sine = np.where((n_dt >= 0) & (n_dt <= 10), np.sin(2 * np.pi * 0.1 * n_dt), 0.0)
delta_origin = np.where(n_dt == 0, 1.0, 0.0)
delta_shifted = np.where(n_dt == 8, 1.0, 0.0) # Delta shifted by 8 samples

y_sine_delta0 = np.convolve(x_sine, delta_origin, mode='same')
y_sine_delta8 = np.convolve(x_sine, delta_shifted, mode='same')

# Plotting Benchmarks
fig, axs = plt.subplots(3, 2, figsize=(13, 11))

# Ex 1
axs[0, 0].plot(t_y1, y_tri_rect, 'b-', linewidth=2)
axs[0, 0].set_title("1. conv(Triangle, Rectangular)")
axs[0, 0].set_ylabel("Amplitude")

# Ex 2
axs[0, 1].plot(t_y1, y_rect_equal, 'g-', linewidth=2)
axs[0, 1].set_title(r"2. conv(Rect, Rect) Equal Widths $\rightarrow$ Triangle")
axs[0, 1].set_ylabel("Amplitude")

# Ex 3
axs[1, 0].plot(t_y1, y_rect_unequal, 'm-', linewidth=2)
axs[1, 0].set_title(r"3. conv(Rect, Rect) Unequal Widths $\rightarrow$ Trapezoid")
axs[1, 0].set_xlabel("Time $t$")
axs[1, 0].set_ylabel("Amplitude")

# Ex 4
axs[1, 1].stem(n_dt, y_sine_delta0, linefmt='c-', markerfmt='co', basefmt='k-')
axs[1, 1].set_title(r"4. conv(Truncated Sine, $\delta[n]$) $\rightarrow$ Identity")
axs[1, 1].set_xlabel("Sample $n$")
axs[1, 1].set_ylabel("Amplitude")

# Ex 5
axs[2, 0].stem(n_dt, y_sine_delta8, linefmt='r-', markerfmt='ro', basefmt='k-')
axs[2, 0].set_title(r"5. conv(Truncated Sine, $\delta[n - 8]$) $\rightarrow$ Shifted Sine")
axs[2, 0].set_xlabel("Sample $n$")
axs[2, 0].set_ylabel("Amplitude")

# Blank off subplot 2,1
axs[2, 1].axis('off')

plt.suptitle("Section 7.2: Built-in 1D Convolution Benchmark Examples", fontsize=14, fontweight='bold')
plt.show()


---
## SECTION 8: Fundamental Properties of Convolution

Linear convolution possesses critical mathematical properties that simplify system design and cascade analysis.

### Properties & Mathematical Identities:

1. **Commutative Property**:
   $$x(t) * h(t) = h(t) * x(t)$$
   The roles of input signal and system impulse response are completely interchangeable.

2. **Shift Property**:
   If $y(t) = x(t) * h(t)$, then shifting either input delays the output by the exact same amount:
   $$x(t - t_0) * h(t) = x(t) * h(t - t_0) = (x * h)(t - t_0)$$

Below, we compute both sides of each equation numerically, report maximum absolute error $\|y_1 - y_2\|_\infty$, and plot superimposed outputs to confirm identity.


In [ ]:
# Section 8: Code Implementation
dt = 0.01
t_vec = np.linspace(0, 4, 400)

# Define test signals
x_sec8 = np.where((t_vec >= 0.5) & (t_vec <= 2.0), np.sin(np.pi * (t_vec - 0.5)), 0.0) # Sine lobe
h_sec8 = np.where((t_vec >= 0.0) & (t_vec <= 1.0), 1.0, 0.0) # Rectangular pulse

# -------------------------------------------------------------
# 1. Verify Commutative Property: x * h == h * x
# -------------------------------------------------------------
conv_x_h = np.convolve(x_sec8, h_sec8, mode='full') * dt
conv_h_x = np.convolve(h_sec8, x_sec8, mode='full') * dt

max_err_comm = np.max(np.abs(conv_x_h - conv_h_x))
print(f"1. Commutative Property Max Absolute Error ||x*h - h*x||: {max_err_comm:.2e}")

# -------------------------------------------------------------
# 2. Verify Shift Property: x(t - t0) * h(t) == (x * h)(t - t0)
# -------------------------------------------------------------
t0 = 1.0 # Delay of 1 second
shift_samples = int(t0 / dt)

# Shift x(t) by t0
x_shifted_sec8 = np.roll(x_sec8, shift_samples)
x_shifted_sec8[:shift_samples] = 0.0

# Convolution of shifted x with original h
conv_shifted_input = np.convolve(x_shifted_sec8, h_sec8, mode='full') * dt

# Shift the original convolution result (x * h) by t0
conv_orig_shifted = np.roll(conv_x_h, shift_samples)
conv_orig_shifted[:shift_samples] = 0.0

max_err_shift = np.max(np.abs(conv_shifted_input - conv_orig_shifted))
print(f"2. Shift Property Max Absolute Error: {max_err_shift:.2e}")

# Plotting Verifications
t_out = np.linspace(0, 8, len(conv_x_h))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Commutative Plot
ax1.plot(t_out, conv_x_h, 'b-', linewidth=3, alpha=0.7, label=r'$x(t) * h(t)$')
ax1.plot(t_out, conv_h_x, 'r:', linewidth=2, label=r'$h(t) * x(t)$')
ax1.set_title(f"Commutative Property (Max Diff = {max_err_comm:.1e})", fontsize=12, fontweight='bold')
ax1.set_xlabel("Time $t$")
ax1.set_ylabel("Amplitude")
ax1.legend()

# Shift Property Plot
ax2.plot(t_out, conv_shifted_input, 'g-', linewidth=3, alpha=0.7, label=r'$x(t - t_0) * h(t)$')
ax2.plot(t_out, conv_orig_shifted, 'm:', linewidth=2, label=r'$(x * h)(t - t_0)$')
ax2.set_title(f"Shift Property (Max Diff = {max_err_shift:.1e})", fontsize=12, fontweight='bold')
ax2.set_xlabel("Time $t$")
ax2.set_ylabel("Amplitude")
ax2.legend()

plt.suptitle("Section 8: Fundamental Properties of Convolution Verification", fontsize=14, fontweight='bold')
plt.show()
